# Comparison: All Approaches

**Цель:** Сравнить все подходы по метрикам и обосновать выбор финальной модели.

**Подходы:**
1. CV Baseline (contour-based)
2. YOLOv8n (single-stage CNN)
3. Faster R-CNN (two-stage CNN)
4. Hybrid (best CNN + CV refiner)


## 0. Colab Setup

⚠️ **Запустить только один раз!** Клонирует репозиторий (sparse checkout) и устанавливает зависимости.

In [ ]:
%%bash
cd /content
rm -rf aie-group-2-sapar
git init aie-group-2-sapar
cd aie-group-2-sapar
git sparse-checkout set project
git remote add origin https://github.com/Sapar-hub/aie-group-2-sapar.git
git pull origin main
cd project
pip install -q ultralytics opencv-python-headless pyyaml pandas matplotlib

In [ ]:
%cd /content/aie-group-2-sapar/project

In [ ]:
!nvidia-smi

## 1. Imports & Load Results

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
METRICS_DIR = ARTIFACTS_DIR / "metrics"
FIGURES_DIR = ARTIFACTS_DIR / "figures"

results = {}
for f in sorted(METRICS_DIR.glob("*.json")):
    with open(f) as fp:
        results[f.stem] = json.load(fp)
    print(f"Loaded: {f.name}")

## 2. Comparison Table

In [ ]:
rows = []
for name, data in results.items():
    rows.append({
        "Model": data.get("model", name),
        "IoU mean": data.get("iou_mean", "-"),
        "IoU std": data.get("iou_std", "-"),
        "Precision": data.get("precision", "-"),
        "Recall": data.get("recall", "-"),
        "F1": data.get("f1", "-"),
        "Det. rate": data.get("detection_rate", "-"),
        "Train (min)": data.get("train_time_min", "-"),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 3. Bar Chart: F1 Score

In [ ]:
plt.figure(figsize=(10, 6))
models = [r["Model"] for r in rows]
f1_scores = [r["F1"] for r in rows]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]
bars = plt.bar(models, f1_scores, color=colors[:len(models)], edgecolor="black")
plt.ylabel("F1 Score")
plt.title("F1 Score Comparison: All Approaches")
plt.ylim(0, 1.0)
for bar, score in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{score:.3f}", ha='center', va='bottom', fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "comparison_f1.png", dpi=150)
plt.show()

## 4. Bar Chart: IoU Mean

In [ ]:
plt.figure(figsize=(10, 6))
iou_means = [r["IoU mean"] for r in rows]
bars = plt.bar(models, iou_means, color=colors[:len(models)], edgecolor="black")
plt.ylabel("Mean IoU")
plt.title("Mean IoU Comparison: All Approaches")
plt.ylim(0, 1.0)
for bar, score in zip(bars, iou_means):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{score:.3f}", ha='center', va='bottom', fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "comparison_iou.png", dpi=150)
plt.show()

## 5. Conclusions & Model Selection

In [ ]:
best_f1_idx = np.argmax([r["F1"] for r in rows])
best_model = rows[best_f1_idx]

print("=" * 50)
print("FINAL MODEL SELECTION")
print("=" * 50)
print(f"Best model: {best_model['Model']}")
print(f"F1 Score:   {best_model['F1']}")
print(f"IoU mean:   {best_model['IoU mean']}")
print(f"Det. rate:  {best_model['Det. rate']}")
print()
print("Justification:")
for i, r in enumerate(rows):
    if i == best_f1_idx:
        print(f"  ★ {r['Model']}: F1={r['F1']}, IoU={r['IoU mean']}")
    else:
        print(f"    {r['Model']}: F1={r['F1']}, IoU={r['IoU mean']}")

summary = {
    "best_model": best_model["Model"],
    "best_f1": best_model["F1"],
    "best_iou": best_model["IoU mean"],
    "all_models": rows,
}

with open(METRICS_DIR / "comparison_results.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\nSaved to {METRICS_DIR / 'comparison_results.json'}")

## 6. Save Results to Git

⚠️ **Запустить после выполнения!** Сохраняет метрики и графики в репозиторий.

In [ ]:
%%bash
cd /content/aie-group-2-sapar
git config user.email "183649607+Sapar-hub@users.noreply.github.com"
git config user.name "Saparmyrat"
git add project/artifacts/metrics/ project/artifacts/figures/
git commit -m "exp07: Final comparison and model selection"
git push origin main